# Experiment: CatBoost on undersampled data, status 404 only

Variant of the CatBoost baseline trained on `train_sampled_data.csv` restricted to status-404 requests.

In [ ]:
DATA_DIR = "../data"
SUBMISSION_DIR = "../data/submissions"

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score

import catboost as cb
# import lightgbm as lgb
# import xgboost as xgb

import tensorflow as tf
from tensorflow.keras import backend as K

## Load Data

In [4]:
ori_train_df = pd.read_csv(f"{DATA_DIR}/train_sampled_data.csv")
ori_train_df.head()

,Unnamed: 0,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,...,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip,rule.mitre.tactic,hour,pairwise
0,0,38,2023-08-29 17:05:43.445,66.96.233.60,-,vm-director-nginx,30/Aug/2023:00:05:42 +0700,POST /lib/ajax/service.php?sesskey=FyapD11LyJ&...,200,98,...,0.03,10.45.184.157:32401,200,688312591,181,66.96.233.60,192.168.1.99,"[""Impact""]",17,0.126820
1,1,54,2023-08-29 17:08:23.939,118.99.125.4,-,vm-director-nginx,30/Aug/2023:00:08:23 +0700,POST /lib/ajax/service.php?sesskey=QY0HDC3szQ&...,200,1169,...,0.068,10.45.184.143:32401,200,688315946,158,118.99.125.4,192.168.1.99,"[""Impact""]",17,0.091207
2,2,98,2023-08-29 17:16:56.322,125.166.1.20,-,vm-director-nginx,30/Aug/2023:00:16:54 +0700,POST /lib/ajax/service.php?sesskey=9dbAEW9y6f&...,200,27,...,0.088,10.45.184.157:32401,200,688328194,209,125.166.1.20,192.168.1.99,"[""Impact""]",17,0.290300
3,3,99,2023-08-29 17:17:14.250,103.171.147.123,-,vm-director-nginx,30/Aug/2023:00:17:13 +0700,POST /lib/ajax/service.php?sesskey=nscrep2nd3&...,200,27,...,0.098,10.45.184.157:32401,200,688330190,90,103.171.147.123,192.168.1.99,"[""Impact""]",17,0.328679
4,4,187,2023-08-29 17:22:58.771,180.248.17.197,-,vm-director-nginx,30/Aug/2023:00:22:56 +0700,POST /lib/ajax/service.php?sesskey=ezddPBRNs5&...,200,6712,...,0.104,10.45.184.142:32401,200,688336653,219,180.248.17.197,192.168.1.99,"[""Impact""]",17,0.077081


In [8]:
ori_train_df = ori_train_df[ori_train_df['status'] == 404]

## Feature Engineering

In [9]:
ori_train_df.upstream_connect_time.value_counts()

0           1510
0.001       1295
0.002         34
0.004          3
-, 0.000       3
0.008          3
0.024          1
0.006          1
0.003          1
0.012          1
Name: upstream_connect_time, dtype: int64

In [10]:
ori_train_df.upstream_response_time.value_counts()

0.002    1311
0.001     756
0.003     437
0          89
0.004      45
         ... 
0.444       1
0.462       1
0.349       1
0.344       1
0.674       1
Name: upstream_response_time, Length: 109, dtype: int64

In [12]:
def clean_upstream_connect_time(x):
    x = x.split(', ')
    for i in range(len(x)):
        try:
            x[i] = float(x[i])
        except ValueError:
            x[i] = 0
    return np.max(x)

def extract_domain_from_url(url):
    pattern = r'^(?:https?:\/\/)?(?:[^@\/\n]+@)?(?:www\.)?([^:\/\n]+)'
    match = re.search(pattern, url)

    if match:
        domain = match.group(1)
        return domain
    else:
        return None
    
def group_http_referer(x):
    x = extract_domain_from_url(x)
    if "notrealdomain" in x:
        return 'notrealdomain'
    elif "175" in x:
        return 'ipaddress'
    elif ".com" in x or '.co.id' in x:
        return 'realwebsite'
    else:
        return x

train_df = ori_train_df.copy()
# train_df['timestamp'] = pd.to_datetime(train_df['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
train_df['upstream_connect_time'] = train_df['upstream_connect_time'].apply(clean_upstream_connect_time)
train_df['upstream_header_time'] = train_df['upstream_header_time'].apply(clean_upstream_connect_time)
train_df['upstream_response_time'] = train_df['upstream_response_time'].apply(clean_upstream_connect_time)
train_df['http_referer'] = train_df['http_referer'].apply(group_http_referer)
# train_df.drop(train_df[train_df['rule.mitre.tactic'] == '["Credential Access"]'].index, inplace=True)
train_df.drop([
    'id',
    'Unnamed: 0',
    'remote_addr',
    'remote_user',
    'hostname',
    'time_local',
    'request',
    'http_referer',
    'http_referer2',
    'http_x_forwarded_for',
    'http_user_agent',
    'time_iso8601',
    'upstream_addr',
    'upstream_status',
    # 'connection',
    'data.srcip',
    'agent.ip',
    'pairwise'
], axis=1, inplace=True)
cat_cols = ['status', 'request_method', 'http_referer']

In [13]:
cat_cols = ['status', 'request_method']

In [14]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2852 entries, 36 to 4757
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   timestamp               2852 non-null   datetime64[ns]
 1   status                  2852 non-null   int64         
 2   body_bytes_sent         2852 non-null   int64         
 3   request_method          2852 non-null   object        
 4   request_length          2852 non-null   int64         
 5   bytes_sent              2852 non-null   int64         
 6   request_time            2852 non-null   float64       
 7   upstream_connect_time   2852 non-null   float64       
 8   upstream_header_time    2852 non-null   float64       
 9   upstream_response_time  2852 non-null   float64       
 10  connection              2852 non-null   int64         
 11  connection_requests     2852 non-null   int64         
 12  rule.mitre.tactic       2852 non-null   object 

In [15]:
train_df.head()

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,rule.mitre.tactic,hour
36,2023-08-29 21:29:41.014,404,2353,GET,575,3074,0.018,0.001,0.017,0.017,688489692,1,"[""Defense Evasion""]",21
42,2023-08-29 21:10:55.738,404,27,GET,338,260,0.180,0.000,0.001,0.001,688475648,3,"[""Reconnaissance""]",21
43,2023-08-29 21:10:59.768,404,8552,GET,607,8963,0.984,0.000,0.010,0.010,688475689,2,"[""Reconnaissance""]",21
44,2023-08-29 21:11:03.754,404,8552,GET,611,8963,0.206,0.001,0.012,0.012,688475689,14,"[""Reconnaissance""]",21
45,2023-08-29 21:10:47.736,404,281,GET,399,456,0.191,0.000,0.000,0.000,688475545,16,"[""Reconnaissance""]",21


In [16]:
train_df.upstream_connect_time.value_counts()

0.000    1513
0.001    1295
0.002      34
0.004       3
0.008       3
0.024       1
0.006       1
0.003       1
0.012       1
Name: upstream_connect_time, dtype: int64

## Modeling

In [17]:
X = train_df.drop('rule.mitre.tactic', axis=1)
y = train_df['rule.mitre.tactic']

In [18]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2852 entries, 36 to 4757
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   timestamp               2852 non-null   datetime64[ns]
 1   status                  2852 non-null   int64         
 2   body_bytes_sent         2852 non-null   int64         
 3   request_method          2852 non-null   object        
 4   request_length          2852 non-null   int64         
 5   bytes_sent              2852 non-null   int64         
 6   request_time            2852 non-null   float64       
 7   upstream_connect_time   2852 non-null   float64       
 8   upstream_header_time    2852 non-null   float64       
 9   upstream_response_time  2852 non-null   float64       
 10  connection              2852 non-null   int64         
 11  connection_requests     2852 non-null   int64         
 12  hour                    2852 non-null   int64  

In [19]:
cbc = cb.CatBoostClassifier(task_type="GPU",
                            cat_features=cat_cols,
                            verbose=1, 
                            random_seed=42)
cbc.fit(X, y)

Learning rate set to 0.080377
0:	learn: 0.9562678	total: 7.81ms	remaining: 7.8s
1:	learn: 0.8436748	total: 14.4ms	remaining: 7.2s
2:	learn: 0.7511897	total: 21ms	remaining: 6.99s
3:	learn: 0.6736263	total: 27.6ms	remaining: 6.88s
4:	learn: 0.6069064	total: 33.8ms	remaining: 6.72s
5:	learn: 0.5502596	total: 41ms	remaining: 6.79s
6:	learn: 0.5000052	total: 47.6ms	remaining: 6.76s
7:	learn: 0.4567129	total: 55ms	remaining: 6.82s
8:	learn: 0.4178238	total: 62.7ms	remaining: 6.91s
9:	learn: 0.3831502	total: 69.5ms	remaining: 6.88s
10:	learn: 0.3525249	total: 76.4ms	remaining: 6.87s
11:	learn: 0.3252441	total: 83.4ms	remaining: 6.87s
12:	learn: 0.3004096	total: 90.5ms	remaining: 6.87s
13:	learn: 0.2785320	total: 99ms	remaining: 6.97s
14:	learn: 0.2581318	total: 107ms	remaining: 7.05s
15:	learn: 0.2398032	total: 116ms	remaining: 7.11s
16:	learn: 0.2232741	total: 123ms	remaining: 7.08s
17:	learn: 0.2085859	total: 132ms	remaining: 7.18s
18:	learn: 0.1945708	total: 141ms	remaining: 7.26s
19:	lea

## Submission

In [20]:
ori_test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
ori_test_df.head()

,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip
0,54481,"Sep 5, 2023 @ 00:26:44.850",10.26.152.242,-,vm-director-nginx,05/Sep/2023:07:26:43 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.002,0.000,0.001,0.001,10.45.184.154:32442,404,721725502,2,10.26.152.242,192.168.1.99
1,54482,"Sep 5, 2023 @ 00:26:44.880",202.80.215.96,-,vm-director-nginx,05/Sep/2023:07:26:44 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.002,0.001,0.003,0.003,10.45.184.148:32442,404,721725753,2,202.80.215.96,192.168.1.99
2,54483,"Sep 5, 2023 @ 00:26:46.898",36.85.73.24,-,vm-director-nginx,05/Sep/2023:07:26:44 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.003,0.001,0.002,0.002,10.45.184.148:32442,404,721725865,2,36.85.73.24,192.168.1.99
3,54484,"Sep 5, 2023 @ 00:26:46.945",180.248.23.157,-,vm-director-nginx,05/Sep/2023:07:26:46 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.002,0.000,0.002,0.002,10.45.184.148:32442,404,721725740,2,180.248.23.157,192.168.1.99
4,54485,"Sep 5, 2023 @ 00:26:48.856",182.1.80.255,-,vm-director-nginx,05/Sep/2023:07:26:46 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,-,...,0.001,0.000,0.001,0.001,10.45.184.154:32442,404,721726205,2,182.1.80.255,192.168.1.99


In [22]:
ori_test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13599 entries, 0 to 13598
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      13599 non-null  int64  
 1   timestamp               13599 non-null  object 
 2   remote_addr             13599 non-null  object 
 3   remote_user             13599 non-null  object 
 4   hostname                13599 non-null  object 
 5   time_local              13599 non-null  object 
 6   request                 13599 non-null  object 
 7   status                  13599 non-null  int64  
 8   body_bytes_sent         13599 non-null  int64  
 9   http_referer            13599 non-null  object 
 10  http_user_agent         13599 non-null  object 
 11  http_x_forwarded_for    13599 non-null  object 
 12  time_iso8601            13599 non-null  object 
 13  request_method          13599 non-null  object 
 14  request_length          13599 non-null

In [31]:
train_df.status.value_counts()

404    2852
Name: status, dtype: int64

In [32]:
ori_test_df.status.value_counts()

404    12722
200      656
400      152
422       25
502       22
500       11
504        6
416        2
403        1
405        1
407        1
Name: status, dtype: int64

In [24]:
test_df = ori_test_df.copy()
test_df['timestamp'] = pd.to_datetime(test_df['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')
# test_df['timestamp'] = pd.to_datetime(test_df['timestamp'])
test_df['upstream_connect_time'] = test_df['upstream_connect_time'].apply(clean_upstream_connect_time)
test_df['upstream_header_time'] = test_df['upstream_header_time'].apply(clean_upstream_connect_time)
test_df['upstream_response_time'] = test_df['upstream_response_time'].apply(clean_upstream_connect_time)
# test_df['http_referer'] = test_df['http_referer'].apply(group_http_referer)
test_df['hour'] = test_df['timestamp'].dt.hour
test_df.drop([
    'id',
    # 'Unnamed: 0',
    'remote_addr',
    'remote_user',
    'hostname',
    'time_local',
    'request',
    'http_referer',
    'http_referer2',
    'http_x_forwarded_for',
    'http_user_agent',
    'time_iso8601',
    'upstream_addr',
    'upstream_status',
    # 'connection',
    'data.srcip',
    'agent.ip',
    # 'pairwise'
], axis=1, inplace=True)
test_df.head()

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,hour
0,2023-09-05 00:26:44.850,404,118,GET,80,330,0.002,0.000,0.001,0.001,721725502,2,0
1,2023-09-05 00:26:44.880,404,118,GET,83,330,0.002,0.001,0.003,0.003,721725753,2,0
2,2023-09-05 00:26:46.898,404,118,GET,85,330,0.003,0.001,0.002,0.002,721725865,2,0
3,2023-09-05 00:26:46.945,404,118,GET,80,330,0.002,0.000,0.002,0.002,721725740,2,0
4,2023-09-05 00:26:48.856,404,118,GET,84,330,0.001,0.000,0.001,0.001,721726205,2,0


In [25]:
y_test_pred = cbc.predict(test_df)
y_test_pred

array([['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ...,
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]']], dtype=object)

In [26]:
y_test_pred[test_df[test_df.status != 404].index]

array([['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Defense Evasion"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Defense Evasion"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Defense Evasion"]'],
       ['["Defense Evasion"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
       ['["Reconnaissance"]'],
    

In [27]:
test_df[test_df.status != 404].head(10)

,timestamp,status,body_bytes_sent,request_method,request_length,bytes_sent,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,connection,connection_requests,hour
43,2023-09-05 00:28:43.454,200,228,POST,154,412,0.035,0.001,0.035,0.035,721742964,139,0
45,2023-09-05 00:28:45.237,200,2274,POST,620,2515,0.279,0.001,0.273,0.279,721742545,15,0
54,2023-09-05 00:28:35.301,200,102,POST,288,333,0.063,0.001,0.061,0.061,721742545,7,0
58,2023-09-05 00:28:13.486,200,5722,POST,302,5931,0.139,0.000,0.137,0.138,721737354,174,0
78,2023-09-05 00:28:55.139,200,167,POST,469,351,0.078,0.001,0.047,0.047,721742997,173,0
98,2023-09-05 00:29:07.108,400,78,POST,1380,473,0.005,0.001,0.006,0.006,721747503,1,0
113,2023-09-05 00:30:07.838,200,100,POST,325,284,0.040,0.000,0.037,0.037,721755939,175,0
148,2023-09-05 00:30:19.748,200,27,POST,270,210,0.159,0.000,0.088,0.088,721763354,45,0
150,2023-09-05 00:30:53.567,200,27,POST,160,210,0.066,0.001,0.066,0.066,721749840,211,0
152,2023-09-05 00:30:55.409,200,166,POST,470,350,0.093,0.001,0.038,0.038,721767373,185,0


In [29]:
submission_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
submission_df['rule.mitre.tactic'] = y_test_pred
submission_df.head()

,id,rule.mitre.tactic
0,54481,"[""Reconnaissance""]"
1,54482,"[""Reconnaissance""]"
2,54483,"[""Reconnaissance""]"
3,54484,"[""Reconnaissance""]"
4,54485,"[""Reconnaissance""]"


In [30]:
submission_df['rule.mitre.tactic'].value_counts()

["Reconnaissance"]     13258
["Defense Evasion"]      339
["non-attack"]             2
Name: rule.mitre.tactic, dtype: int64

In [41]:
train_full = pd.read_csv(f"{DATA_DIR}/train.csv")

In [44]:
cross_tab = pd.crosstab(train_full['http_referer'], train_full['rule.mitre.tactic'])
# Filter rows where only the 'non attack' column has values greater than 0
cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]

# Filter rows where only the 'non attack' column has values greater than 0
non_attack_http = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
non_attack_http_list = non_attack_http.index.tolist()

In [47]:
ori_test_df[ori_test_df['http_referer'].isin(non_attack_http_list)]


,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip
257,54738,"Sep 5, 2023 @ 00:31:47.210",10.200.144.115,-,vm-director-nginx,05/Sep/2023:07:31:46 +0700,GET /assets/default/css/xtheme.css?cb6766e HTT...,404,281,https://halofilkom.notrealdomain.com/open.php,...,0.001,0.001,0.001,0.001,10.45.184.59:8001,404,721783562,3,10.200.144.115,192.168.1.99
521,55002,"Sep 5, 2023 @ 00:35:55.517",36.71.145.232,-,vm-director-nginx,05/Sep/2023:07:35:55 +0700,GET /assets/default/css/xtheme.css?cb6766e HTT...,404,281,https://halofilkom.notrealdomain.com/kb/faq.ph...,...,0.001,0.001,0.002,0.002,10.45.184.59:8001,404,721825599,3,36.71.145.232,192.168.1.99
1400,55881,"Sep 5, 2023 @ 00:55:12.999",10.200.160.31,-,vm-director-nginx,05/Sep/2023:07:55:12 +0700,GET /ujian.php?key=MzA7MTk7MDA7MDI7MTsyMDIzOzE...,404,36,https://siam.notrealdomain.com/exam.php,...,0.002,0.000,0.000,0.000,175.45.184.244:80,404,721970221,20,10.200.160.31,192.168.1.99
1419,55900,"Sep 5, 2023 @ 00:55:02.907",10.200.160.31,-,vm-director-nginx,05/Sep/2023:07:55:01 +0700,GET /ujian.php?key=MzA7MTk7MDA7MDI7MTsyMDIzOzE...,404,36,https://siam.notrealdomain.com/exam.php,...,0.002,0.000,0.001,0.001,175.45.184.244:80,404,721970221,19,10.200.160.31,192.168.1.99
1486,55967,"Sep 5, 2023 @ 00:57:00.966",10.34.3.151,-,vm-director-nginx,05/Sep/2023:07:57:00 +0700,GET /assets/default/css/xtheme.css?cb6766e HTT...,404,281,https://halofilkom.notrealdomain.com/,...,0.001,0.001,0.002,0.002,10.45.184.59:8001,404,721986851,2,10.34.3.151,192.168.1.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13201,67682,"Sep 6, 2023 @ 02:03:57.362",118.99.112.12,-,vm-director-nginx,06/Sep/2023:09:03:56 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2425,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.011,0.000,0.010,0.010,175.45.185.12:80,404,729170418,40,118.99.112.12,192.168.1.99
13438,67919,"Sep 6, 2023 @ 02:09:55.723",182.1.67.189,-,vm-director-nginx,06/Sep/2023:09:09:54 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2421,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.012,0.000,0.011,0.011,175.45.185.12:80,404,729216416,41,182.1.67.189,192.168.1.99
13483,67964,"Sep 6, 2023 @ 02:10:59.831",110.138.213.129,-,vm-director-nginx,06/Sep/2023:09:10:59 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2424,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.012,0.000,0.012,0.012,175.45.185.12:80,404,729237155,16,110.138.213.129,192.168.1.99
13493,67974,"Sep 6, 2023 @ 02:11:03.777",182.1.67.189,-,vm-director-nginx,06/Sep/2023:09:11:02 +0700,GET /kkn/kkn-kelompok-lokasi/undefined/undefin...,404,2423,https://mmd.notrealdomain.com/kkn/kkn-kelompok...,...,0.015,0.001,0.014,0.015,175.45.185.12:80,404,729216416,54,182.1.67.189,192.168.1.99


In [54]:
ori_test_df[ori_test_df['request_method'].isin(["HEAD"])] 


,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,http_referer,...,request_time,upstream_connect_time,upstream_header_time,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip
192,54673,"Sep 5, 2023 @ 00:31:19.288",180.248.43.56,-,vm-director-nginx,05/Sep/2023:07:31:17 +0700,HEAD /https://iam.ub.ac.id/ HTTP/1.1,404,0,-,...,0.002,0.001,0.002,0.002,10.45.184.154:32442,404,721779079,1,180.248.43.56,192.168.1.99
6027,60508,"Sep 5, 2023 @ 22:34:53.144",202.80.214.69,-,vm-director-nginx,06/Sep/2023:05:34:52 +0700,HEAD /https://iam.ub.ac.id/ HTTP/1.1,404,0,-,...,0.002,0.000,0.001,0.001,10.45.184.154:32442,404,728024745,1,202.80.214.69,192.168.1.99
8725,63206,"Sep 6, 2023 @ 00:21:34.290",10.200.145.39,-,vm-director-nginx,06/Sep/2023:07:21:32 +0700,HEAD /https://iam.ub.ac.id/ HTTP/1.1,404,0,-,...,0.001,0.000,0.001,0.001,10.45.184.154:32442,404,728392396,1,10.200.145.39,192.168.1.99


In [34]:
submission_df.to_csv(f"{SUBMISSION_DIR}/submission1.csv", index=False)

OSError: Cannot save file into a non-existent directory: 'submissions'

## Post Evaluation

In [ ]:

# Filter rows where only the 'non attack' column has values greater than 0
non_attack_http = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
non_attack_http_list = non_attack_http.index.tolist()

test[test['http_referer'].isin(non_attack_http_list)] = ["non-attack"]
